# PDEBench-Lang: T5 Baseline for PDE Classification

**Goal:** First-pass notebook baseline using T5-small with natural language dialect input.

**Target:** Single concatenated seq2seq output containing:
- Behavioral label (PDE family)
- Operators
- Reasoning chain

**Budget:** 3 epochs on full dataset

## Phase 1: Data/Task Setup

In [25]:
# Install required packages
!pip install -q transformers datasets evaluate rouge_score scikit-learn

In [26]:
import json
import random
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import evaluate

# Set reproducible config
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Configuration
CONFIG = {
    "model_name": "t5-small",
    "max_input_length": 128,
    "max_output_length": 256,
    "learning_rate": 3e-4,
    "batch_size": 16,
    "num_epochs": 3,
    "val_split": 0.1,
    "seed": SEED,
}

print(f"Configuration: {CONFIG}")
print(f"CUDA available: {torch.cuda.is_available()}")

Configuration: {'model_name': 't5-small', 'max_input_length': 128, 'max_output_length': 256, 'learning_rate': 0.0003, 'batch_size': 16, 'num_epochs': 3, 'val_split': 0.1, 'seed': 42}
CUDA available: True


In [27]:
# Load dataset and validate schema
def load_dataset_jsonl(filepath):
    """Load dataset from JSONL file and validate required fields."""
    data = []
    required_fields = ["family", "dialects", "labels"]

    with open(filepath, "r") as f:
        for line_num, line in enumerate(f, 1):
            instance = json.loads(line.strip())

            # Validate schema
            for field in required_fields:
                if field not in instance:
                    raise ValueError(f"Line {line_num}: Missing required field '{field}'")

            # Validate nested fields
            if "natural" not in instance["dialects"]:
                raise ValueError(f"Line {line_num}: Missing 'dialects.natural'")
            if "behavioral" not in instance["labels"]:
                raise ValueError(f"Line {line_num}: Missing 'labels.behavioral'")
            if "operators" not in instance["labels"]:
                raise ValueError(f"Line {line_num}: Missing 'labels.operators'")
            if "reasoning" not in instance["labels"]:
                raise ValueError(f"Line {line_num}: Missing 'labels.reasoning'")

            data.append(instance)

    return data

# Load the dataset
dataset_path = "dataset.jsonl"
raw_data = load_dataset_jsonl(dataset_path)

print(f"Loaded {len(raw_data)} instances")
print(f"\nSample instance:")
print(json.dumps(raw_data[0], indent=2))

Loaded 10000 instances

Sample instance:
{
  "family": "Heat",
  "coefficients": {
    "alpha": 1.8
  },
  "dialects": {
    "latex": "\\frac{\\partial}{\\partial t} u{\\left(x,t \\right)} = 1.8 \\frac{\\partial^{2}}{\\partial x^{2}} u{\\left(x,t \\right)}",
    "prefix": "= d(u,t) *(1.8 d(d(u,x),x))",
    "postfix": "u t d 1.8 u x x d d * =",
    "natural": "The time derivative of u equals 1.8 times the second spatial derivative of u."
  },
  "labels": {
    "behavioral": "Heat",
    "operators": [
      "exp",
      "polynomial"
    ],
    "reasoning": "The equation contains a 1st-order time derivative (u_t) on the left hand side and a 2nd-order spatial derivative (u_xx) on the right hand side. The coefficient alpha = 1.8 scales the spatial diffusion term. This structure is characteristic of diffusive transport \u2014 spatial curvature drives temporal change. The equation belongs to the Heat/Diffusion family."
  }
}


In [28]:
# Analyze family distribution
from collections import Counter

family_counts = Counter(instance["family"] for instance in raw_data)
print("PDE Family Distribution:")
print("-" * 30)
for family, count in sorted(family_counts.items()):
    print(f"{family}: {count} ({count/len(raw_data)*100:.1f}%)")

PDE Family Distribution:
------------------------------
Advection: 2000 (20.0%)
Burgers: 2000 (20.0%)
Heat: 2000 (20.0%)
Laplace: 2000 (20.0%)
Wave: 2000 (20.0%)


In [29]:
# Define stable input/output templates
def format_input(instance):
    """Format input using natural language dialect."""
    natural_text = instance["dialects"]["natural"]
    return f"classify PDE: {natural_text}"

def format_output(instance):
    """Format target output as concatenated sequence."""
    behavioral = instance["labels"]["behavioral"]
    operators = ", ".join(instance["labels"]["operators"])
    reasoning = instance["labels"]["reasoning"]

    return f"Family: {behavioral} | Operators: {operators} | Reasoning: {reasoning}"

# Preview templates
sample = raw_data[0]
print("Input template:")
print(format_input(sample))
print("\nOutput template:")
print(format_output(sample))

Input template:
classify PDE: The time derivative of u equals 1.8 times the second spatial derivative of u.

Output template:
Family: Heat | Operators: exp, polynomial | Reasoning: The equation contains a 1st-order time derivative (u_t) on the left hand side and a 2nd-order spatial derivative (u_xx) on the right hand side. The coefficient alpha = 1.8 scales the spatial diffusion term. This structure is characteristic of diffusive transport — spatial curvature drives temporal change. The equation belongs to the Heat/Diffusion family.


## Phase 2: Training Pipeline

In [30]:
# Build stratified train/validation split by family
labels = [instance["family"] for instance in raw_data]

train_data, val_data = train_test_split(
    raw_data,
    test_size=CONFIG["val_split"],
    stratify=labels,
    random_state=CONFIG["seed"]
)

print(f"Train size: {len(train_data)}")
print(f"Validation size: {len(val_data)}")

# Verify balanced distribution
train_families = Counter(d["family"] for d in train_data)
val_families = Counter(d["family"] for d in val_data)

print("\nTrain distribution:")
for family, count in sorted(train_families.items()):
    print(f"  {family}: {count}")

print("\nValidation distribution:")
for family, count in sorted(val_families.items()):
    print(f"  {family}: {count}")

Train size: 9000
Validation size: 1000

Train distribution:
  Advection: 1800
  Burgers: 1800
  Heat: 1800
  Laplace: 1800
  Wave: 1800

Validation distribution:
  Advection: 200
  Burgers: 200
  Heat: 200
  Laplace: 200
  Wave: 200


In [31]:
# Initialize tokenizer and model
tokenizer = T5Tokenizer.from_pretrained(CONFIG["model_name"])
model = T5ForConditionalGeneration.from_pretrained(CONFIG["model_name"])

print(f"Model loaded: {CONFIG['model_name']}")
print(f"Model parameters: {model.num_parameters():,}")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Model loaded: t5-small
Model parameters: 60,506,624


In [32]:
# Preprocess and tokenize
def preprocess_data(data):
    """Convert raw data to formatted input/output pairs."""
    return {
        "input_text": [format_input(d) for d in data],
        "target_text": [format_output(d) for d in data],
        "family": [d["family"] for d in data],
    }

def tokenize_function(examples):
    """Tokenize inputs and targets."""
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=CONFIG["max_input_length"],
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        examples["target_text"],
        max_length=CONFIG["max_output_length"],
        truncation=True,
        padding="max_length",
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Create datasets
train_processed = preprocess_data(train_data)
val_processed = preprocess_data(val_data)

train_dataset = Dataset.from_dict(train_processed)
val_dataset = Dataset.from_dict(val_processed)

# Tokenize
train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=["input_text", "target_text", "family"])
val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=["input_text", "target_text", "family"])

print(f"Train dataset: {len(train_tokenized)} examples")
print(f"Validation dataset: {len(val_tokenized)} examples")

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train dataset: 9000 examples
Validation dataset: 1000 examples


In [33]:
# Check truncation rates
def check_truncation(data, max_input, max_output):
    """Check how many examples exceed token limits."""
    input_truncated = 0
    output_truncated = 0

    for instance in data:
        input_tokens = tokenizer(format_input(instance), return_length=True)["length"][0]
        output_tokens = tokenizer(format_output(instance), return_length=True)["length"][0]

        if input_tokens > max_input:
            input_truncated += 1
        if output_tokens > max_output:
            output_truncated += 1

    return input_truncated, output_truncated

input_trunc, output_trunc = check_truncation(
    raw_data,
    CONFIG["max_input_length"],
    CONFIG["max_output_length"]
)

print(f"Input truncation: {input_trunc}/{len(raw_data)} ({input_trunc/len(raw_data)*100:.2f}%)")
print(f"Output truncation: {output_trunc}/{len(raw_data)} ({output_trunc/len(raw_data)*100:.2f}%)")

Input truncation: 0/10000 (0.00%)
Output truncation: 0/10000 (0.00%)


In [34]:
# Setup training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_pde_results",
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    predict_with_generate=True,
    generation_max_length=CONFIG["max_output_length"],
    logging_dir="./logs",
    logging_steps=100,
    seed=CONFIG["seed"],
    report_to="none",
)

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [35]:
# Initialize trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Trainer initialized successfully")

Trainer initialized successfully


In [36]:
# Train the model
print("Starting training...")
train_result = trainer.train()

print("\nTraining complete!")
print(f"Training loss: {train_result.training_loss:.4f}")

Starting training...


Epoch,Training Loss,Validation Loss
1,0.001173,0.000235
2,0.000448,0.000096
3,0.000308,0.000030


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].



Training complete!
Training loss: 0.0329


## Phase 3: Evaluation

In [37]:
# Generate predictions on validation set
def generate_predictions(model, tokenizer, data, batch_size=16):
    """Generate predictions for all examples."""
    model.eval()
    predictions = []

    device = next(model.parameters()).device

    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        inputs = [format_input(d) for d in batch]

        encoded = tokenizer(
            inputs,
            max_length=CONFIG["max_input_length"],
            truncation=True,
            padding=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **encoded,
                max_length=CONFIG["max_output_length"],
                num_beams=4,
                early_stopping=True,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend(decoded)

    return predictions

print("Generating predictions on validation set...")
val_predictions = generate_predictions(model, tokenizer, val_data)
print(f"Generated {len(val_predictions)} predictions")

Generating predictions on validation set...
Generated 1000 predictions


In [38]:
# Parse predictions into components
import re

def parse_prediction(pred_str):
    """Parse prediction string into family, operators, and reasoning."""
    result = {
        "family": None,
        "operators": [],
        "reasoning": None,
    }

    # Extract family
    family_match = re.search(r"Family:\s*([^|]+)", pred_str)
    if family_match:
        result["family"] = family_match.group(1).strip()

    # Extract operators
    ops_match = re.search(r"Operators:\s*([^|]+)", pred_str)
    if ops_match:
        ops_str = ops_match.group(1).strip()
        result["operators"] = [op.strip() for op in ops_str.split(",") if op.strip()]

    # Extract reasoning
    reason_match = re.search(r"Reasoning:\s*(.+)$", pred_str)
    if reason_match:
        result["reasoning"] = reason_match.group(1).strip()

    return result

# Parse all predictions
parsed_predictions = [parse_prediction(p) for p in val_predictions]

# Show sample
print("Sample prediction parsing:")
print(f"Raw: {val_predictions[0]}")
print(f"Parsed: {parsed_predictions[0]}")

Sample prediction parsing:
Raw: Family: Burgers | Operators: tanh, polynomial | Reasoning: The equation contains a 1st-order time derivative (u_t) and a nonlinear convective term (u * u_x) on the left hand side, and a 2nd-order spatial derivative (u_xx) scaled by viscosity nu = 1.71 on the right hand side. The nonlinear convection balanced against diffusion is characteristic of the Burgers equation family.
Parsed: {'family': 'Burgers', 'operators': ['tanh', 'polynomial'], 'reasoning': 'The equation contains a 1st-order time derivative (u_t) and a nonlinear convective term (u * u_x) on the left hand side, and a 2nd-order spatial derivative (u_xx) scaled by viscosity nu = 1.71 on the right hand side. The nonlinear convection balanced against diffusion is characteristic of the Burgers equation family.'}


In [39]:
# Compute evaluation metrics
rouge = evaluate.load("rouge")

def compute_metrics(val_data, parsed_predictions, val_predictions):
    """Compute all evaluation metrics."""
    metrics = {}

    # 1. Behavioral Exact Match (Family Accuracy)
    correct_family = sum(
        1 for d, p in zip(val_data, parsed_predictions)
        if p["family"] and p["family"].lower() == d["labels"]["behavioral"].lower()
    )
    metrics["family_accuracy"] = correct_family / len(val_data)

    # 2. Operator Set F1
    total_precision = 0
    total_recall = 0
    valid_ops = 0

    for d, p in zip(val_data, parsed_predictions):
        true_ops = set(op.lower() for op in d["labels"]["operators"])
        pred_ops = set(op.lower() for op in p["operators"])

        if pred_ops:
            precision = len(true_ops & pred_ops) / len(pred_ops)
            total_precision += precision
        if true_ops:
            recall = len(true_ops & pred_ops) / len(true_ops)
            total_recall += recall
            valid_ops += 1

    avg_precision = total_precision / len(val_data) if val_data else 0
    avg_recall = total_recall / valid_ops if valid_ops else 0

    if avg_precision + avg_recall > 0:
        metrics["operator_f1"] = 2 * avg_precision * avg_recall / (avg_precision + avg_recall)
    else:
        metrics["operator_f1"] = 0

    metrics["operator_precision"] = avg_precision
    metrics["operator_recall"] = avg_recall

    # 3. Reasoning ROUGE-L
    pred_reasoning = [p["reasoning"] or "" for p in parsed_predictions]
    true_reasoning = [d["labels"]["reasoning"] for d in val_data]

    rouge_scores = rouge.compute(
        predictions=pred_reasoning,
        references=true_reasoning
    )
    metrics["reasoning_rouge_l"] = rouge_scores["rougeL"]

    return metrics

metrics = compute_metrics(val_data, parsed_predictions, val_predictions)

print("="*50)
print("EVALUATION METRICS")
print("="*50)
print(f"Family Accuracy:     {metrics['family_accuracy']:.4f}")
print(f"Operator Precision:  {metrics['operator_precision']:.4f}")
print(f"Operator Recall:     {metrics['operator_recall']:.4f}")
print(f"Operator F1:         {metrics['operator_f1']:.4f}")
print(f"Reasoning ROUGE-L:   {metrics['reasoning_rouge_l']:.4f}")
print("="*50)

EVALUATION METRICS
Family Accuracy:     1.0000
Operator Precision:  1.0000
Operator Recall:     1.0000
Operator F1:         1.0000
Reasoning ROUGE-L:   0.9943


In [40]:
# Per-family accuracy breakdown
from collections import defaultdict

family_results = defaultdict(lambda: {"correct": 0, "total": 0})

for d, p in zip(val_data, parsed_predictions):
    true_family = d["family"]
    pred_family = p["family"] or "NONE"

    family_results[true_family]["total"] += 1
    if pred_family.lower() == true_family.lower():
        family_results[true_family]["correct"] += 1

print("Per-Family Accuracy:")
print("-" * 40)
for family, results in sorted(family_results.items()):
    acc = results["correct"] / results["total"] if results["total"] > 0 else 0
    print(f"{family:15} {results['correct']:4}/{results['total']:4} ({acc:.2%})")

Per-Family Accuracy:
----------------------------------------
Advection        200/ 200 (100.00%)
Burgers          200/ 200 (100.00%)
Heat             200/ 200 (100.00%)
Laplace          200/ 200 (100.00%)
Wave             200/ 200 (100.00%)


In [41]:
# Qualitative prediction inspection across all families
print("="*80)
print("QUALITATIVE INSPECTION: Sample Predictions per Family")
print("="*80)

families_seen = set()

for i, (d, pred_str, parsed) in enumerate(zip(val_data, val_predictions, parsed_predictions)):
    family = d["family"]
    if family in families_seen:
        continue
    families_seen.add(family)

    print(f"\n--- {family} ---")
    print(f"Input: {format_input(d)[:100]}...")
    print(f"\nTarget Family: {d['labels']['behavioral']}")
    print(f"Predicted Family: {parsed['family']}")
    print(f"\nTarget Operators: {d['labels']['operators']}")
    print(f"Predicted Operators: {parsed['operators']}")
    print(f"\nTarget Reasoning (truncated): {d['labels']['reasoning'][:150]}...")
    print(f"Predicted Reasoning (truncated): {(parsed['reasoning'] or '')[:150]}...")

    is_correct = parsed['family'] and parsed['family'].lower() == d['labels']['behavioral'].lower()
    print(f"\n✓ Correct" if is_correct else "✗ Incorrect")

print("\n" + "="*80)

QUALITATIVE INSPECTION: Sample Predictions per Family

--- Burgers ---
Input: classify PDE: The time derivative of u plus u times the spatial derivative of u equals 1.71 times th...

Target Family: Burgers
Predicted Family: Burgers

Target Operators: ['tanh', 'polynomial']
Predicted Operators: ['tanh', 'polynomial']

Target Reasoning (truncated): The equation contains a 1st-order time derivative (u_t) and a nonlinear convective term (u * u_x) on the left hand side, and a 2nd-order spatial deriv...
Predicted Reasoning (truncated): The equation contains a 1st-order time derivative (u_t) and a nonlinear convective term (u * u_x) on the left hand side, and a 2nd-order spatial deriv...

✓ Correct

--- Heat ---
Input: classify PDE: The time derivative of u equals 1.24 times the second spatial derivative of u....

Target Family: Heat
Predicted Family: Heat

Target Operators: ['exp', 'polynomial']
Predicted Operators: ['exp', 'polynomial']

Target Reasoning (truncated): The equation contains a

In [42]:
# Log common failure modes
print("="*60)
print("FAILURE MODE ANALYSIS")
print("="*60)

# 1. Wrong family predictions
wrong_family = []
for d, p in zip(val_data, parsed_predictions):
    if not p["family"] or p["family"].lower() != d["labels"]["behavioral"].lower():
        wrong_family.append({
            "true": d["labels"]["behavioral"],
            "pred": p["family"],
            "input": d["dialects"]["natural"][:80]
        })

print(f"\n1. Wrong Family Predictions: {len(wrong_family)}/{len(val_data)}")
if wrong_family:
    confusion = defaultdict(lambda: defaultdict(int))
    for item in wrong_family:
        confusion[item["true"]][item["pred"] or "NONE"] += 1

    print("\nConfusion patterns (True -> Predicted):")
    for true_fam, preds in sorted(confusion.items()):
        for pred_fam, count in sorted(preds.items(), key=lambda x: -x[1])[:3]:
            print(f"  {true_fam} -> {pred_fam}: {count}")

# 2. Operator hallucination
all_valid_operators = {"exp", "polynomial", "sin", "cos", "tanh"}
hallucinated_ops = defaultdict(int)

for p in parsed_predictions:
    for op in p["operators"]:
        if op.lower() not in all_valid_operators:
            hallucinated_ops[op.lower()] += 1

print(f"\n2. Operator Hallucinations:")
if hallucinated_ops:
    for op, count in sorted(hallucinated_ops.items(), key=lambda x: -x[1])[:5]:
        print(f"  '{op}': {count} occurrences")
else:
    print("  None detected")

# 3. Missing reasoning
missing_reasoning = sum(1 for p in parsed_predictions if not p["reasoning"])
print(f"\n3. Missing Reasoning Chains: {missing_reasoning}/{len(val_data)}")

# 4. Parsing failures
parse_failures = sum(
    1 for p in parsed_predictions
    if not p["family"] and not p["operators"] and not p["reasoning"]
)
print(f"\n4. Complete Parse Failures: {parse_failures}/{len(val_data)}")

FAILURE MODE ANALYSIS

1. Wrong Family Predictions: 0/1000

2. Operator Hallucinations:
  None detected

3. Missing Reasoning Chains: 0/1000

4. Complete Parse Failures: 0/1000


## Phase 4: Exploratory Test Results (50 Predictions)

In [43]:
# Select 50 stratified samples for exploratory analysis
import pandas as pd
from IPython.display import display, HTML

# Get 10 samples per family for balanced representation
exploratory_indices = []
family_counts_exp = defaultdict(int)
samples_per_family = 10

for i, d in enumerate(val_data):
    family = d["family"]
    if family_counts_exp[family] < samples_per_family:
        exploratory_indices.append(i)
        family_counts_exp[family] += 1
    if len(exploratory_indices) >= 50:
        break

print(f"Selected {len(exploratory_indices)} samples for exploratory analysis")
print(f"Distribution: {dict(family_counts_exp)}")

Selected 50 samples for exploratory analysis
Distribution: {'Burgers': 10, 'Heat': 10, 'Laplace': 10, 'Advection': 10, 'Wave': 10}


In [44]:
# Create detailed results table for 50 predictions
detailed_results = []

for idx in exploratory_indices:
    d = val_data[idx]
    p = parsed_predictions[idx]
    raw_pred = val_predictions[idx]

    # Check correctness
    family_correct = p["family"] and p["family"].lower() == d["labels"]["behavioral"].lower()

    true_ops = set(op.lower() for op in d["labels"]["operators"])
    pred_ops = set(op.lower() for op in p["operators"])
    ops_match = true_ops == pred_ops

    detailed_results.append({
        "#": len(detailed_results) + 1,
        "True Family": d["labels"]["behavioral"],
        "Pred Family": p["family"] or "NONE",
        "Family ✓": "✓" if family_correct else "✗",
        "True Ops": ", ".join(d["labels"]["operators"]),
        "Pred Ops": ", ".join(p["operators"]) if p["operators"] else "NONE",
        "Ops ✓": "✓" if ops_match else "~" if (true_ops & pred_ops) else "✗",
        "Input (truncated)": d["dialects"]["natural"][:60] + "...",
    })

# Display as DataFrame
df_detailed = pd.DataFrame(detailed_results)

print("="*100)
print("DETAILED PREDICTION RESULTS (50 Samples)")
print("="*100)
print("Legend: ✓ = correct, ✗ = wrong, ~ = partial match")
print()

# Style the dataframe for better readability
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', None)
display(df_detailed)

DETAILED PREDICTION RESULTS (50 Samples)
Legend: ✓ = correct, ✗ = wrong, ~ = partial match



,#,True Family,Pred Family,Family ✓,True Ops,Pred Ops,Ops ✓,Input (truncated)
0,1,Burgers,Burgers,✓,"tanh, polynomial","tanh, polynomial",✓,The time derivative of u plus u times the spatial derivative...
1,2,Heat,Heat,✓,"exp, polynomial","exp, polynomial",✓,The time derivative of u equals 1.24 times the second spatia...
2,3,Burgers,Burgers,✓,"tanh, polynomial","tanh, polynomial",✓,The time derivative of u plus u times the spatial derivative...
3,4,Laplace,Laplace,✓,"sin, cos, exp, polynomial","sin, cos, exp, polynomial",✓,0.87 times the sum of the second spatial derivative of u in ...
4,5,Advection,Advection,✓,"exp, polynomial","exp, polynomial",✓,The time derivative of u plus 1.83 times the spatial derivat...
5,6,Advection,Advection,✓,"exp, polynomial","exp, polynomial",✓,The time derivative of u plus 1.69 times the spatial derivat...
6,7,Wave,Wave,✓,"sin, cos, polynomial","sin, cos, polynomial",✓,The second time derivative of u equals 1.59 times the second...
7,8,Wave,Wave,✓,"sin, cos, polynomial","sin, cos, polynomial",✓,The second time derivative of u equals 0.25 times the second...
8,9,Heat,Heat,✓,"exp, polynomial","exp, polynomial",✓,The time derivative of u equals 0.12 times the second spatia...
9,10,Wave,Wave,✓,"sin, cos, polynomial","sin, cos, polynomial",✓,The second time derivative of u equals 1.19 times the second...


In [45]:
# Create summary statistics table
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

# Overall metrics on full validation set
overall_stats = pd.DataFrame([
    {"Metric": "Family Accuracy", "Value": f"{metrics['family_accuracy']:.2%}", "Description": "Correct PDE family classification"},
    {"Metric": "Operator Precision", "Value": f"{metrics['operator_precision']:.2%}", "Description": "Predicted operators that are correct"},
    {"Metric": "Operator Recall", "Value": f"{metrics['operator_recall']:.2%}", "Description": "True operators that were predicted"},
    {"Metric": "Operator F1", "Value": f"{metrics['operator_f1']:.2%}", "Description": "Harmonic mean of precision/recall"},
    {"Metric": "Reasoning ROUGE-L", "Value": f"{metrics['reasoning_rouge_l']:.2%}", "Description": "Reasoning chain overlap score"},
])

print("\n📊 Overall Metrics (Full Validation Set):")
display(overall_stats)

# Per-family breakdown
per_family_stats = pd.DataFrame([
    {
        "Family": family,
        "Correct": results["correct"],
        "Total": results["total"],
        "Accuracy": f"{results['correct']/results['total']:.2%}" if results["total"] > 0 else "N/A"
    }
    for family, results in sorted(family_results.items())
])

print("\n📈 Per-Family Accuracy:")
display(per_family_stats)

# Exploratory sample stats
exp_family_correct = sum(1 for r in detailed_results if r["Family ✓"] == "✓")
exp_ops_exact = sum(1 for r in detailed_results if r["Ops ✓"] == "✓")
exp_ops_partial = sum(1 for r in detailed_results if r["Ops ✓"] == "~")

exploratory_stats = pd.DataFrame([
    {"Metric": "Family Correct", "Count": exp_family_correct, "Rate": f"{exp_family_correct/50:.2%}"},
    {"Metric": "Operators Exact Match", "Count": exp_ops_exact, "Rate": f"{exp_ops_exact/50:.2%}"},
    {"Metric": "Operators Partial Match", "Count": exp_ops_partial, "Rate": f"{exp_ops_partial/50:.2%}"},
    {"Metric": "Operators Wrong", "Count": 50 - exp_ops_exact - exp_ops_partial, "Rate": f"{(50 - exp_ops_exact - exp_ops_partial)/50:.2%}"},
])

print("\n🔬 Exploratory Sample Stats (50 Predictions):")
display(exploratory_stats)


SUMMARY STATISTICS

📊 Overall Metrics (Full Validation Set):


,Metric,Value,Description
0,Family Accuracy,100.00%,Correct PDE family classification
1,Operator Precision,100.00%,Predicted operators that are correct
2,Operator Recall,100.00%,True operators that were predicted
3,Operator F1,100.00%,Harmonic mean of precision/recall
4,Reasoning ROUGE-L,99.43%,Reasoning chain overlap score



📈 Per-Family Accuracy:


,Family,Correct,Total,Accuracy
0,Advection,200,200,100.00%
1,Burgers,200,200,100.00%
2,Heat,200,200,100.00%
3,Laplace,200,200,100.00%
4,Wave,200,200,100.00%



🔬 Exploratory Sample Stats (50 Predictions):


,Metric,Count,Rate
0,Family Correct,50,100.00%
1,Operators Exact Match,50,100.00%
2,Operators Partial Match,0,0.00%
3,Operators Wrong,0,0.00%


In [46]:
# Confusion matrix for family predictions
print("\n" + "="*60)
print("FAMILY CONFUSION MATRIX (Full Validation Set)")
print("="*60)

families = sorted(set(d["family"] for d in val_data))
confusion = defaultdict(lambda: defaultdict(int))

for d, p in zip(val_data, parsed_predictions):
    true_fam = d["family"]
    pred_fam = p["family"] or "NONE"
    confusion[true_fam][pred_fam] += 1

# Build confusion matrix DataFrame
all_pred_families = sorted(set(p["family"] or "NONE" for p in parsed_predictions))
cm_data = []
for true_fam in families:
    row = {"True \\ Pred": true_fam}
    for pred_fam in all_pred_families:
        row[pred_fam] = confusion[true_fam][pred_fam]
    cm_data.append(row)

df_confusion = pd.DataFrame(cm_data)
display(df_confusion)


FAMILY CONFUSION MATRIX (Full Validation Set)


,True \ Pred,Advection,Burgers,Heat,Laplace,Wave
0,Advection,200,0,0,0,0
1,Burgers,0,200,0,0,0
2,Heat,0,0,200,0,0
3,Laplace,0,0,0,200,0
4,Wave,0,0,0,0,200


In [47]:
# Final run configuration summary
print("\n" + "="*60)
print("RUN CONFIGURATION")
print("="*60)

config_summary = pd.DataFrame([
    {"Parameter": "Model", "Value": CONFIG["model_name"]},
    {"Parameter": "Input Dialect", "Value": "Natural Language"},
    {"Parameter": "Training Epochs", "Value": CONFIG["num_epochs"]},
    {"Parameter": "Batch Size", "Value": CONFIG["batch_size"]},
    {"Parameter": "Learning Rate", "Value": CONFIG["learning_rate"]},
    {"Parameter": "Max Input Length", "Value": CONFIG["max_input_length"]},
    {"Parameter": "Max Output Length", "Value": CONFIG["max_output_length"]},
    {"Parameter": "Train Size", "Value": len(train_data)},
    {"Parameter": "Validation Size", "Value": len(val_data)},
    {"Parameter": "Final Training Loss", "Value": f"{train_result.training_loss:.4f}"},
])

display(config_summary)


RUN CONFIGURATION


,Parameter,Value
0,Model,t5-small
1,Input Dialect,Natural Language
2,Training Epochs,3
3,Batch Size,16
4,Learning Rate,0.0003
5,Max Input Length,128
6,Max Output Length,256
7,Train Size,9000
8,Validation Size,1000
9,Final Training Loss,0.0329


## Conclusion

This prototype notebook establishes a T5-small baseline for PDEBench-Lang using natural language input.

**Results logged above:**
- 📋 Detailed prediction table (50 samples across all families)
- 📊 Overall metrics (accuracy, F1, ROUGE-L)
- 📈 Per-family accuracy breakdown
- 🔀 Confusion matrix
- ⚙️ Run configuration

**Note:** This is a lightweight prototype for sharing. No model checkpoint is saved.

**Next steps (out of scope for this prototype):**
- Cross-dialect comparison (Postfix, LaTeX, Prefix)
- External symbolic solver pruning integration
- Advanced adaptation methods (LoRA, multi-head variants)
- Model checkpoint saving for production use